In [3]:
import os

experiment_dir = "/home/al.herivelton.siqueira/Downloads/ml2_final_project/experiments/tacotron2-vae/v1_tacotron_verbo"

print("--- Conteúdo principal da pasta do experimento ---")
if os.path.exists(experiment_dir):
    arquivos = os.listdir(experiment_dir)
    for arq in arquivos:
        print(f" Solto na pasta: {arq}")
        
    # Se houver uma subpasta de checkpoints, vamos olhar dentro dela
    checkpoints_dir = os.path.join(experiment_dir, "checkpoints")
    if os.path.exists(checkpoints_dir):
        print("\n--- Conteúdo da subpasta 'checkpoints' ---")
        for ckpt in os.listdir(checkpoints_dir):
            print(f"  Dentro de checkpoints: {ckpt}")
else:
    print(f"Erro: O caminho {experiment_dir} não foi encontrado no sistema.")

--- Conteúdo principal da pasta do experimento ---
 Solto na pasta: training_plots
 Solto na pasta: symbols.json
 Solto na pasta: tensorboard
 Solto na pasta: hparams.json
 Solto na pasta: checkpoints
 Solto na pasta: logs

--- Conteúdo da subpasta 'checkpoints' ---
  Dentro de checkpoints: epoch_0


In [12]:
import os

src_dir = "/home/al.herivelton.siqueira/Downloads/ml2_final_project/src"

print("=== Mapeando a estrutura do diretório 'src' ===")
if os.path.exists(src_dir):
    for root, dirs, files in os.walk(src_dir):
        # Filtra para não poluir a tela com pastas temporárias do python (__pycache__)
        if "__pycache__" in root or ".ipynb_checkpoints" in root:
            continue
            
        # Verifica se algum arquivo de texto ou pasta relevante está aqui
        for f in files:
            if "text" in f or "sequence" in f or f.endswith(".py"):
                # Mostra o caminho relativo a partir de 'src'
                rel_path = os.path.relpath(os.path.join(root, f), src_dir)
                print(f" Encontrado arquivo: src/{rel_path}")
else:
    print(f"Erro: O caminho {src_dir} não foi encontrado.")

=== Mapeando a estrutura do diretório 'src' ===
 Encontrado arquivo: src/models/GST.py
 Encontrado arquivo: src/models/tacotron2_vae/utils.py
 Encontrado arquivo: src/models/tacotron2_vae/stft.py
 Encontrado arquivo: src/models/tacotron2_vae/hparams.py
 Encontrado arquivo: src/models/tacotron2_vae/audio_processing.py
 Encontrado arquivo: src/models/tacotron2_vae/model.py
 Encontrado arquivo: src/models/tacotron2_vae/__init__.py
 Encontrado arquivo: src/models/tacotron2_vae/modules.py
 Encontrado arquivo: src/models/tacotron2_vae/coord_conv.py
 Encontrado arquivo: src/models/tacotron2_vae/layers.py
 Encontrado arquivo: src/models/waveglow/glow.py
 Encontrado arquivo: src/data/__init__.py
 Encontrado arquivo: src/data/loader_vae_tacotron/test.py
 Encontrado arquivo: src/data/loader_vae_tacotron/loader_tacotron.py
 Encontrado arquivo: src/data/loader_waveglow/loader_waveglow.py
 Encontrado arquivo: src/training/__init__.py
 Encontrado arquivo: src/training/training-waveglow/train.py
 Enco

In [1]:
import sys
import os
import json
import numpy as np
import torch

# 1. Garante que o Jupyter use o ambiente virtual correto com o PyTorch
venv_path = "/home/al.herivelton.siqueira/Downloads/ml2_final_project/.venv/lib/python3.12/site-packages"
if venv_path not in sys.path:
    sys.path.insert(0, venv_path)

# 2. Configura a raiz do projeto e caminhos de busca do Python
ROOT_DIR = "/home/al.herivelton.siqueira/Downloads/ml2_final_project"
os.chdir(ROOT_DIR)

SRC_DIR = os.path.join(ROOT_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

training_dir = os.path.join(SRC_DIR, "training", "training-tacotron2-vae")
if training_dir not in sys.path:
    sys.path.insert(0, training_dir)

print("[✔] Ambiente configurado com os caminhos corretos.")

# 3. Importações do Modelo e do Processador de Texto local
from models.tacotron2_vae.model import Tacotron2
from models.tacotron2_vae.hparams import create_hparams
import text_processing

# 4. Carregar o checkpoint para coletar as dimensões exatas do treino
checkpoint_path = f"{ROOT_DIR}/experiments/tacotron2-vae/v1_tacotron_verbo/checkpoints/epoch_0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

checkpoint = torch.load(checkpoint_path, map_location=device)
print("[✔] Checkpoint carregado com sucesso.")

# 5. Configurar os Hparams usando os dados exatos do checkpoint
hparams = create_hparams()
if "hparams" in checkpoint:
    for k, v in checkpoint["hparams"].items():
        if hasattr(hparams, k):
            setattr(hparams, k, v)
    print("[✔] Hyperparameters restaurados do checkpoint.")
else:
    hparams_json_path = f"{ROOT_DIR}/experiments/tacotron2-vae/v1_tacotron_verbo/hparams.json"
    if os.path.exists(hparams_json_path):
        with open(hparams_json_path, 'r') as f:
            data = json.load(f)
            for k, v in data.items():
                if hasattr(hparams, k):
                    setattr(hparams, k, v)
        print("[✔] Hyperparameters carregados do hparams.json.")

# 6. Instanciar o modelo Tacotron2 e aplicar os pesos salvos
model = Tacotron2(hparams).to(device)
state_dict_key = 'model_state_dict' if 'model_state_dict' in checkpoint else 'state_dict'
model.load_state_dict(checkpoint[state_dict_key])
model.eval() 
print("[✔] Pesos do Tacotron2 acoplados com sucesso!")

# 7. Conversão ULTRA SEGURA de texto
# Usaremos uma frase extremamente simples, toda em minúsculas e sem pontuação pontual
texto_para_falar = "teste de sintese de voz"

try:
    symbols_json_path = f"{ROOT_DIR}/experiments/tacotron2-vae/v1_tacotron_verbo/symbols.json"
    if os.path.exists(symbols_json_path):
        text_processor = text_processing.TextProcessor.load(symbols_json_path)
    else:
        symbols = getattr(hparams, 'symbols', text_processing.DEFAULT_SYMBOLS)
        text_processor = text_processing.TextProcessor(symbols)
    
    # Obtém a sequência numérica pura
    raw_sequence = text_processor.text_to_sequence(texto_para_falar)
    vocab_size = model.transcript_embedding.weight.shape[0]
    
    # CLIP DE SEGURANÇA: Filtra qualquer ID que seja maior ou igual ao tamanho do vocabulário do modelo
    safe_sequence = [id_sym if id_sym < vocab_size else 1 for id_sym in raw_sequence]
    
    sequence = torch.LongTensor(safe_sequence).unsqueeze(0).to(device)
    transcript_embedded_inputs = model.transcript_embedding(sequence).transpose(1, 2)
    print(f"[✔] Texto convertido com segurança! IDs enviados para a GPU: {safe_sequence}")
    
    # 8. Execução da Pipeline Manual de Inferência
    print(f"\nSintetizando: '{texto_para_falar}'...")
    with torch.no_grad():
        # Encoder
        transcript_outputs = model.encoder.inference(transcript_embedded_inputs)
        
        # Vetor de Estilo Neutro (Zeros)
        n_mel_channels = getattr(hparams, 'n_mel_channels', 80)
        fake_reference_mel = torch.zeros(1, n_mel_channels, 100).to(device)
        
        latent_vector, _, _, _ = model.vae_gst(fake_reference_mel)
        latent_vector = latent_vector.unsqueeze(1).expand_as(transcript_outputs)
        
        # Combinação e Decoder
        encoder_outputs = transcript_outputs + latent_vector
        model.decoder.max_decoder_steps = 1000
        mel_outputs, gate_outputs, alignments = model.decoder.inference(encoder_outputs)
        
        # PostNet
        mel_outputs_postnet = model.postnet(mel_outputs)
        mel_outputs_postnet = mel_outputs + mel_outputs_postnet
        
        print("[✔] Pipeline concluído! Espectrograma Mel gerado com sucesso.")
        print(f"👉 Formato do espectrograma final: {mel_outputs_postnet.shape}")

except Exception as e:
    print(f"[❌] Erro ao executar passos da pipeline: {e}")

[✔] Ambiente configurado com os caminhos corretos.


/home/al.herivelton.siqueira/Downloads/ml2_final_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[✔] Checkpoint carregado com sucesso.
[✔] Hyperparameters restaurados do checkpoint.
[✔] Pesos do Tacotron2 acoplados com sucesso!
[✔] Texto convertido com segurança! IDs enviados para a GPU: [30, 16, 29, 30, 16, 11, 15, 16, 11, 29, 20, 24, 30, 16, 29, 16, 11, 15, 16, 11, 32, 25, 34, 43]

Sintetizando: 'teste de sintese de voz'...
Warning! Reached max decoder steps
[✔] Pipeline concluído! Espectrograma Mel gerado com sucesso.
👉 Formato do espectrograma final: torch.Size([1, 80, 1000])


In [5]:
import os

raiz_projeto = "/home/al.herivelton.siqueira/Downloads/ml2_final_project/"
print("=== Buscando o arquivo HiFi_GAN no projeto ===")

encontrado = False
for raiz, diretorios, arquivos in os.walk(raiz_projeto):
    for arquivo in arquivos:
        if "HiFi_GAN" in arquivo or "hifi_gan" in arquivo.lower():
            caminho_completo = os.path.join(raiz, arquivo)
            print(f"🔍 Encontrado em: {caminho_completo}")
            encontrado = True

if not encontrado:
    print("❌ Nenhum arquivo com o nome HiFi_GAN foi localizado nas subpastas.")

=== Buscando o arquivo HiFi_GAN no projeto ===
🔍 Encontrado em: /home/al.herivelton.siqueira/Downloads/ml2_final_project/env/lib64/python3.12/site-packages/torchaudio/prototype/models/hifi_gan.py
🔍 Encontrado em: /home/al.herivelton.siqueira/Downloads/ml2_final_project/env/lib64/python3.12/site-packages/torchaudio/prototype/models/__pycache__/hifi_gan.cpython-312.pyc


In [6]:
import sys
import os
import torch
import torchaudio
import torchaudio.transforms as T

# 1. Configuração de caminhos e dispositivos
ROOT_DIR = "/home/al.herivelton.siqueira/Downloads/ml2_final_project"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=== Convertendo Espectrograma Mel para Áudio (.wav) ===")

# 2. Carrega o espectrograma Mel salvo pelo Tacotron2
mel_path = f"{ROOT_DIR}/mel_output_epoch0.pt"

if os.path.exists(mel_path):
    # Carrega os dados (Formato esperado: Batch x Canais_Mel x Frames)
    mel_spectrogram = torch.load(mel_path, map_location=device)
    print(f"[✔] Espectrograma Mel carregado com sucesso! Formato: {mel_spectrogram.shape}")
    
    # 3. Configuração do Reconstrutor de Áudio (Griffin-Lim)
    # Definindo parâmetros acústicos padrão que casam com o Tacotron 2
    sampling_rate = 22050
    n_fft = 1024
    hop_length = 256
    win_length = 1024
    
    # Criamos a transformação que converte a escala Mel de volta para Espectrograma Linear, 
    # e na sequência reconstrói a fase da onda sonora.
    inverse_mel_transform = T.InverseMelScale(
        n_stft=n_fft // 2 + 1, 
        n_mels=80, 
        sample_rate=sampling_rate
    ).to(device)
    
    griffin_lim = T.GriffinLim(
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        power=1.0,
        n_iter=64 # Número de iterações para refinar a qualidade matemática do áudio
    ).to(device)
    
    try:
        with torch.no_grad():
            # Passo A: Converte de escala Mel logarítmica para espectrograma linear
            # Removendo a dimensão de batch temporariamente para a transformação se necessário
            mel_linear = inverse_mel_transform(mel_spectrogram.squeeze(0))
            
            # Passo B: Aplica Griffin-Lim para reconstruir a forma de onda (.wav)
            waveform = griffin_lim(mel_linear)
            
            # Garante que o formato final seja (Canais x Tempo) adicionando a dimensão de canal de volta
            if waveform.dim() == 1:
                waveform = waveform.unsqueeze(0)
            
            # Envia o resultado para a CPU para salvamento em disco
            waveform_cpu = waveform.cpu()
            
            # Passo C: Salva o arquivo de áudio final
            output_path = f"{ROOT_DIR}/output_final_tacotron2_epoch0.wav"
            torchaudio.save(output_path, waveform_cpu, sampling_rate)
            
            print(f"\n[✔][✔][✔] SUCESSO ABSOLUTO!")
            print(f"👉 Arquivo de áudio gerado com sucesso em:\n📍 {output_path}")
            print("\n💡 Lembrete: Como este modelo pertence à época 0 de treino, o som gerado será um chiado contínuo.")
            print("Isso valida perfeitamente que sua pipeline de dados e áudio está 100% pronta!")
            
    except Exception as e:
        print(f"[❌] Erro durante a reconstrução matemática do áudio: {e}")
else:
    print(f"[❌] Erro: O arquivo de entrada '{mel_path}' não foi localizado na raiz do projeto.")

=== Convertendo Espectrograma Mel para Áudio (.wav) ===
[✔] Espectrograma Mel carregado com sucesso! Formato: torch.Size([1, 80, 1000])

[✔][✔][✔] SUCESSO ABSOLUTO!
👉 Arquivo de áudio gerado com sucesso em:
📍 /home/al.herivelton.siqueira/Downloads/ml2_final_project/output_final_tacotron2_epoch0.wav

💡 Lembrete: Como este modelo pertence à época 0 de treino, o som gerado será um chiado contínuo.
Isso valida perfeitamente que sua pipeline de dados e áudio está 100% pronta!


In [7]:
import sys
import os
import json
import torch
import torchaudio
import torchaudio.transforms as T

# 1. Configuração do ambiente
ROOT_DIR = "/home/al.herivelton.siqueira/Downloads/ml2_final_project"
sys.path.insert(0, os.path.join(ROOT_DIR, "src"))
sys.path.insert(0, os.path.join(ROOT_DIR, "src", "training", "training-tacotron2-vae"))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from models.tacotron2_vae.model import Tacotron2
from models.tacotron2_vae.hparams import create_hparams
import text_processing

print("=== Carregando o modelo da Época 50 ===")

# 2. Localizar o último checkpoint (Verifica se gerou epoch_50 ou o arquivo de checkpoint final)
# Altere para 'epoch_50' ou o nome exato do arquivo que está na sua pasta de checkpoints
checkpoint_dir = f"{ROOT_DIR}/experiments/tacotron2-vae/v1_tacotron_verbo/checkpoints"
print(f"Arquivos disponíveis na pasta: {os.listdir(checkpoint_dir)}")

# Vamos mirar no peso mais recente (ajuste o nome se necessário, ex: 'epoch_49', 'checkpoint_50')
ultimo_checkpoint = sorted(os.listdir(checkpoint_dir))[-1] 
checkpoint_path = os.path.join(checkpoint_dir, ultimo_checkpoint)
print(f"Carregando pesos de: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location=device)

# 3. Restaurar Hparams e Instanciar
hparams = create_hparams()
if "hparams" in checkpoint:
    for k, v in checkpoint["hparams"].items():
        if hasattr(hparams, k): setattr(hparams, k, v)

model = Tacotron2(hparams).to(device)
state_dict_key = 'model_state_dict' if 'model_state_dict' in checkpoint else 'state_dict'
model.load_state_dict(checkpoint[state_dict_key])
model.eval()

# 4. Texto de Teste (Podemos arriscar uma frase levemente maior)
texto_para_falar = "teste de sintese de voz apos cinquenta epocas de treino"

symbols_json_path = f"{ROOT_DIR}/experiments/tacotron2-vae/v1_tacotron_verbo/symbols.json"
text_processor = text_processing.TextProcessor.load(symbols_json_path) if os.path.exists(symbols_json_path) else text_processing.TextProcessor(getattr(hparams, 'symbols', text_processing.DEFAULT_SYMBOLS))

raw_sequence = text_processor.text_to_sequence(texto_para_falar)
vocab_size = model.transcript_embedding.weight.shape[0]
safe_sequence = [id_sym if id_sym < vocab_size else 1 for id_sym in raw_sequence]
sequence = torch.LongTensor(safe_sequence).unsqueeze(0).to(device)

# 5. Pipeline de Inferência Manual
print(f"\nSintetizando: '{texto_para_falar}'...")
with torch.no_grad():
    transcript_embedded_inputs = model.transcript_embedding(sequence).transpose(1, 2)
    transcript_outputs = model.encoder.inference(transcript_embedded_inputs)
    
    fake_reference_mel = torch.zeros(1, getattr(hparams, 'n_mel_channels', 80), 100).to(device)
    latent_vector, _, _, _ = model.vae_gst(fake_reference_mel)
    latent_vector = latent_vector.unsqueeze(1).expand_as(transcript_outputs)
    
    encoder_outputs = transcript_outputs + latent_vector
    model.decoder.max_decoder_steps = 1000
    mel_outputs, gate_outputs, alignments = model.decoder.inference(encoder_outputs)
    
    mel_outputs_postnet = model.postnet(mel_outputs) + mel_outputs

print(f"[✔] Novo espectrograma gerado: {mel_outputs_postnet.shape}")

# 6. Transformar em Áudio via Griffin-Lim
inverse_mel_transform = T.InverseMelScale(n_stft=1024//2 + 1, n_mels=80, sample_rate=22050).to(device)
griffin_lim = T.GriffinLim(n_fft=1024, hop_length=256, win_length=1024, power=1.0, n_iter=64).to(device)

with torch.no_grad():
    mel_linear = inverse_mel_transform(mel_outputs_postnet.squeeze(0))
    waveform = griffin_lim(mel_linear)
    if waveform.dim() == 1: waveform = waveform.unsqueeze(0)
    
    output_path = f"{ROOT_DIR}/output_final_tacotron2_epoch50.wav"
    torchaudio.save(output_path, waveform.cpu(), 22050)
    print(f"\n👉 NOVO ÁUDIO GERADO EM: {output_path}")

=== Carregando o modelo da Época 50 ===
Arquivos disponíveis na pasta: ['epoch_0']
Carregando pesos de: /home/al.herivelton.siqueira/Downloads/ml2_final_project/experiments/tacotron2-vae/v1_tacotron_verbo/checkpoints/epoch_0

Sintetizando: 'teste de sintese de voz apos cinquenta epocas de treino'...
Warning! Reached max decoder steps
[✔] Novo espectrograma gerado: torch.Size([1, 80, 1000])

👉 NOVO ÁUDIO GERADO EM: /home/al.herivelton.siqueira/Downloads/ml2_final_project/output_final_tacotron2_epoch50.wav


In [8]:
print("=== Diagnóstico do Espectrograma Gerado ===")
print(f"Formato do tensor: {mel_outputs_postnet.shape}")
print(f"Valor Máximo: {mel_outputs_postnet.max().item():.4f}")
print(f"Valor Mínimo: {mel_outputs_postnet.min().item():.4f}")
print(f"Média dos valores: {mel_outputs_postnet.mean().item():.4f}")

# Dá uma olhada no que o Gate (mecanismo de parada) está prevendo
print(f"Média das predições do Gate: {gate_outputs.mean().item():.4f}")

=== Diagnóstico do Espectrograma Gerado ===
Formato do tensor: torch.Size([1, 80, 1000])
Valor Máximo: 1.4850
Valor Mínimo: -7.7627
Média dos valores: -3.0978
Média das predições do Gate: -10.1673
